# NF Beam Mapping

A notebook for recovering phase from beam maps in the near-field!

In [1]:
## 20240308 NF/FF Flights with fixed pulsing cadence over WLC... (use WLC Site named for each polarization!)
## Group 1: ch0=Bicolog, ch1=Telescope_Y_NSpol
    ## 'FLY001Airdata.csv' - Altitude ~60m, Yaw ~0deg  --> FFNS
    ## 'FLY002Airdata.csv' - Altitude ~10m, Yaw ~0deg   --> NFNS
## Group 2: ch0=Telescope_X_EWpol, ch1=Bicolog
    ## 'FLY003Airdata.csv' - Altitude ~10m, Yaw ~90deg  --> NFEW
    ## 'FLY004Airdata.csv' - Altitude ~60m, Yaw ~90deg --> FFEW

##From loadD3Adata_Dallas.py:
from matplotlib.pyplot import *
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LogNorm
from matplotlib.ticker import MultipleLocator
import numpy as np
import h5py
##From WT:
import datetime
import pytz
import os
import glob
from matplotlib import colors
import pandas
import csv
import pickle
%matplotlib inline
import beamcals
from beamcals import corr
from beamcals import concat
from beamcals import drone
from beamcals import bicolog
from beamcals import beammap
from beamcals import reduce_ccclass
import beamcals.plotting_utils as pu
import beamcals.fitting_utils as fu
import beamcals.geometry_utils as gu
import beamcals.time_utils as tu
import beamcals.reduce_ccclass as rc
from beamcals.sites import site

## Key Functions

### For fringe fitting and phase unwrapping:

In [2]:
#Define some nice functions for this notebook:

from scipy.interpolate import griddata

def unwrap2d(phi_arr,wrap_args=(False,False)):
    naninds=np.isnan(phi_arr)
    mask_arr=np.zeros_like(phi_arr,dtype=bool)
    mask_arr[naninds]=True    
    phi_mask=np.ma.array(phi_arr,mask=mask_arr)
    phi_mask[naninds]=0.0
    phi_unwrap=unwrap_phase(phi_mask,wrap_around=wrap_args)
    phi_out=phi_unwrap.data-np.nanmean(phi_unwrap.data)
    phi_out[naninds]=np.nan
    phi_arr[naninds]=np.nan
    return(phi_out)

from scipy.optimize import least_squares
from scipy.optimize import curve_fit

# Solve for ABC st (drnpos•[A,B,C])==np.angle(V01) for drnpos in cut boundaries where geometric 
# phase contribution is nonnegligible

def Fringe_Fit(P,dronepos,V):
    A,B,C,D,E,F,G=P
    phase=(np.dot(dronepos+np.array([D,E,F])/np.abs(dronepos+np.array([D,E,F])),[A,B,C]))+G
    return phase-V

def Fringe(P,dronepos):
    A,B,C,D,E,F,G=P
    phase=(np.dot(dronepos+np.array([D,E,F])/np.abs(dronepos+np.array([D,E,F])),[A,B,C]))+G
    return phase

def Fringe_Fit(P,dronepos,V,wavelengthm):
    A,B,C,G=P
    phi=(2.0*np.pi/wavelengthm)*((A*(dronepos[:,0]))+(B*(dronepos[:,1]))+(C*(dronepos[:,2])))
    return phi-V

def Fringe(P,dronepos,wavelengthm):
    A,B,C,G=P
    phi=(2.0*np.pi/wavelengthm)*((A*(dronepos[:,0]))+(B*(dronepos[:,1]))+(C*(dronepos[:,2])))
    return phi

### For restricting the beam (to remove NaNs)

In [3]:
def plot_for_cutoff(beam, find, xargs, yargs):
    
    '''
    xargs=[xmin,xmax]
    yargs=[ymin,ymax]
    '''
    
    fig,[[ax1,ax2,ax3,ax4],[ax5,ax6,ax7,ax8]]=subplots(nrows=2,ncols=4,figsize=(16,9))
    ax1.pcolormesh(beam.d0_centers_grid[:,:,0],beam.d1_centers_grid[:,:,0],beam.V_LC_cross.real[:,:,find,0,0])
    ax2.pcolormesh(beam.d0_centers_grid[:,:,0],beam.d1_centers_grid[:,:,0],beam.V_LC_cross.imag[:,:,find,0,0])
    ax3.pcolormesh(beam.d0_centers_grid[:,:,0],beam.d1_centers_grid[:,:,0],np.abs(beam.V_LC_cross)[:,:,find,0,0])#,norm=LogNorm())
    ax4.pcolormesh(beam.d0_centers_grid[:,:,0],beam.d1_centers_grid[:,:,0],np.angle(beam.V_LC_cross)[:,:,find,0,0])

    ax5.pcolormesh(beam.d0_centers_grid[:,:,0],beam.d1_centers_grid[:,:,0],beam.beam_linear_interp.real[:,:,find,0])
    ax6.pcolormesh(beam.d0_centers_grid[:,:,0],beam.d1_centers_grid[:,:,0],beam.beam_linear_interp.imag[:,:,find,0])
    ax7.pcolormesh(beam.d0_centers_grid[:,:,0],beam.d1_centers_grid[:,:,0],beam.beam_linear_interp_amp[:,:,find,0])#,norm=LogNorm())
    ax8.pcolormesh(beam.d0_centers_grid[:,:,0],beam.d1_centers_grid[:,:,0],beam.beam_linear_interp_phase[:,:,find,0])

    titles=['real','imag','mag','phase','interp_real','interp_imag','interp_mag','interp_phase']
    for i,ax in enumerate([ax1,ax2,ax3,ax4,ax5,ax6,ax7,ax8]):
        ax.set_title(titles[i]+' @ {} --> {:.2f} MHz'.format(find,FFNSbeam.freq[find]))
        ax.set_xlabel('x [m]')
        ax.set_ylabel('y [m]')
        ## Plot the lines where we will do the reassignment/dimension change
        ax.axvline(beam.d0_centers_grid[xargs[0],0,0])
        ax.axhline(beam.d1_centers_grid[0,yargs[0],0])
        ax.axvline(beam.d0_centers_grid[xargs[1],0,0])
        ax.axhline(beam.d1_centers_grid[0,yargs[1],0])

    tight_layout()

In [ ]:
def restrict_beam(beam, xargs, yargs):
    
    [xmin,xmax]=xargs
    [ymin,ymax]=yargs 
    
    beam.beam_linear_interp=beam.beam_linear_interp[xmin:xmax,ymin:ymax]
    beam.beam_linear_interp_amp=beam.beam_linear_interp_amp[xmin:xmax,ymin:ymax]
    beam.beam_linear_interp_phase=beam.beam_linear_interp_phase[xmin:xmax,ymin:ymax]
    beam.beam_linear_interp_phase_unwrapped=beam.beam_linear_interp_phase_unwrapped[xmin:xmax,ymin:ymax]
    beam.histogram_LC=beam.histogram_LC[xmin:xmax,ymin:ymax]
    beam.V_LC_cross=beam.V_LC_cross[xmin:xmax,ymin:ymax]
    beam.V_LC_mean=beam.V_LC_mean[xmin:xmax,ymin:ymax]
    beam.V_LC_operation=beam.V_LC_operation[xmin:xmax,ymin:ymax]
    beam.V_LC_std=beam.V_LC_std[xmin:xmax,ymin:ymax]
    beam.d0_centers_grid=beam.d0_centers_grid[xmin:xmax,ymin:ymax]
    beam.d0_edges_grid=beam.d0_edges_grid[xmin:xmax,ymin:ymax]
    beam.d1_centers_grid=beam.d1_centers_grid[xmin:xmax,ymin:ymax]
    beam.d1_edges_grid=beam.d1_edges_grid[xmin:xmax,ymin:ymax]

    beam.d0_centers=beam.d0_centers[xmin:xmax]
    beam.d0_edges=beam.d0_edges[xmin:xmax]
    beam.d1_centers=beam.d1_centers[ymin:ymax]
    beam.d1_edges=beam.d1_edges[ymin:ymax]